# 3. Pi0 モデルの推論と動画作成
学習済みモデルを用いてGenesis上でゼロショット推論を行い、結果をMP4動画として保存します。
課題の要件である「1分〜3分以内」を満たすようにフレームレートやステップ数を調整しています。


In [1]:
import genesis as gs
import numpy as np
import imageio
import torch
import os
from PIL import Image

# --- エラー回避用のモンキーパッチ ---
if not hasattr(torch._C._CudaDeviceProperties, 'shared_memory_per_multiprocessor'):
    torch._C._CudaDeviceProperties.shared_memory_per_multiprocessor = 102400
if not hasattr(torch._C._CudaDeviceProperties, 'shared_memory_per_block_optin'):
    torch._C._CudaDeviceProperties.shared_memory_per_block_optin = 99328

# 1. MuJoCo Menagerie（ロボット3Dモデル集）のダウンロード
if not os.path.exists("mujoco_menagerie"):
    !git clone https://github.com/google-deepmind/mujoco_menagerie.git

# 2. Genesisの初期化と環境構築
gs.init(backend=gs.cuda) # RTX GPUを利用

scene = gs.Scene(show_viewer=False)
plane = scene.add_entity(gs.morphs.Plane())

# 3. MJCF形式でUR5eを読み込み
robot = scene.add_entity(
    gs.morphs.MJCF(file="mujoco_menagerie/universal_robots_ur5e/ur5e.xml")
)

cam = scene.add_camera(
    res=(640, 480),
    pos=(1.5, 1.5, 1.0),
    lookat=(0, 0, 0.5),
    fov=60,
    GUI=False
)
scene.build()
print('Genesis環境を構築しました。')


e:\matuo-PYZERO\.venv\Lib\site-packages\genesis\__init__.py:41: UserWarning: 'torch<2.8.0' is not supported. Please upgrade pytorch manually: https://pytorch.org/get-started/locally/
  warn("'torch<2.8.0' is not supported. Please upgrade pytorch manually: https://pytorch.org/get-started/locally/")


[Genesis] [21:05:59] [WARNING] 'torch<2.8.0' is not supported. Please upgrade pytorch manually: https://pytorch.org/get-started/locally/
[Genesis] [21:05:59] [INFO] ╭───────────────────────────────────────────────╮
[Genesis] [21:05:59] [INFO] │┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈ Genesis ┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈│
[Genesis] [21:05:59] [INFO] ╰───────────────────────────────────────────────╯
[Genesis] [21:05:59] [INFO] Running on [NVIDIA GeForce RTX 3070 Ti] with backend gs.cuda. Device memory: 8.00 GB.
[Genesis] [21:05:59] [INFO] 🚀 Genesis initialized. 🔖 version: 1.4.5, 🎨 theme: dark, 🌱 seed: None, 🐛 debug: False, 📏 precision: 32, 🔥 performance: False, 💬 verbose: INFO
[Genesis] [21:06:00] [INFO] Scene <6a5f344> created.
[Genesis] [21:06:00] [INFO] Adding <gs.engine.entities.RigidEntity> plane_f3e6652. idx: 0, morph: <gs.morphs.Plane>, material: <gs.materials.Rigid>.
[Genesis] [21:06:01] [WARNING] Mesh is not watertight. Falling back to convex hull for estimating inertial properties.
[Genesis] [21:06:01] [WAR

e:\matuo-PYZERO\.venv\Lib\site-packages\quadrants\lang\ast\ast_transformers\function_def_transformer.py:781: DeprecationWarning: ast.Str is deprecated and will be removed in Python 3.14; use ast.Constant instead
  return index == 0 and isinstance(stmt, ast.Expr) and isinstance(stmt.value, (ast.Constant, ast.Str))


[Genesis] [21:07:57] [INFO] Building visualizer...
Genesis環境を構築しました。


In [2]:
from transformers import AutoModelForVision2Seq, AutoProcessor
from PIL import Image
import torch
import numpy as np

class PreTrainedVLAModel:
    def __init__(self):
        print("学習済みモデル (OpenVLA-7B) をロード中... (初回は14GB程度のDLが発生します)")
        model_id = "openvla/openvla-7b"
        
        self.processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
        
        self.model = AutoModelForVision2Seq.from_pretrained(
            model_id,
            trust_remote_code=True,
            torch_dtype=torch.float16,
            load_in_4bit=True, # VRAM 8GBに収めるための量子化
            device_map="auto"
        )
        print("モデルのロードが完了しました！")

    def predict_action(self, image_np, instruction):
        # 1. 画像の変換 (GenesisのRGB配列 -> PIL)
        image_pil = Image.fromarray(image_np).convert("RGB")
        
        # 2. OpenVLA指定のプロンプトフォーマット
        prompt = f"In: What action should the robot take to {instruction}?\nOut:"
        inputs = self.processor(prompt, image_pil).to("cuda", dtype=torch.float16)
        
        # 3. 推論実行
        with torch.inference_mode():
            # アクションは7次元: [dx, dy, dz, d_roll, d_pitch, d_yaw, gripper]
            action = self.model.predict_action(**inputs, unnorm_key="bridge_orig")
            
        return action

# モデルの初期化と指示のセット
model = PreTrainedVLAModel()
instruction = "move arm forward"

e:\matuo-PYZERO\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


学習済みモデル (OpenVLA-7B) をロード中... (初回は14GB程度のDLが発生します)


e:\matuo-PYZERO\.venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
e:\matuo-PYZERO\.venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
Expected `transformers==4.40.1` and `tokenizers==0.19.1` but got `transformers==4.40.0` and `tokenizers==0.19.1`; there might be inference-time regressions due to dependency changes. If in doubt, pleaseuse the above versions.
The model weigh

モデルのロードが完了しました！


In [3]:
def run_inference_and_record():
    frames = []
    steps = 900 # 30fpsで30秒 (API推論が重いためステップ数を削減)
    
    robot.set_dofs_position(np.zeros(robot.n_dofs))
    
    print('シミュレーションと推論を開始します...')
    for step in range(steps):
        rgb, _, _, _ = cam.render()
        if torch.is_tensor(rgb):
            rgb = rgb.detach().cpu().numpy()
            
        # VLAモデルによる推論 
        vla_action = model.predict_action(rgb, instruction)

        joint_action = vla_action[:6] * 0.5 
        
        robot.control_dofs_velocity(joint_action)
        scene.step()
        
        frames.append(rgb)
        if step > 0 and step % 30 == 0:
            print(f'Step {step}/{steps} completed. ({step//30}秒分録画完了)')
            
    print('動画のレンダリング中...')
    imageio.mimsave('pai_final_task_openvla.mp4', frames, fps=30)
    print('課題への提出準備が完了しました。')

run_inference_and_record()

シミュレーションと推論を開始します...


e:\matuo-PYZERO\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:957: UserWarning: inner dimension (4304) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(


[Genesis] [21:08:40] [INFO] Running at 0.24 FPS.
[Genesis] [21:08:41] [INFO] Running at 0.28 FPS.
[Genesis] [21:08:42] [INFO] Running at 0.32 FPS.
[Genesis] [21:08:43] [INFO] Running at 0.34 FPS.
[Genesis] [21:08:44] [INFO] Running at 0.38 FPS.
[Genesis] [21:08:45] [INFO] Running at 0.42 FPS.
[Genesis] [21:08:46] [INFO] Running at 0.45 FPS.
[Genesis] [21:08:47] [INFO] Running at 0.49 FPS.
[Genesis] [21:08:48] [INFO] Running at 0.53 FPS.
[Genesis] [21:08:49] [INFO] Running at 0.56 FPS.
[Genesis] [21:08:50] [INFO] Running at 0.58 FPS.
[Genesis] [21:08:50] [INFO] Running at 0.61 FPS.
[Genesis] [21:08:51] [INFO] Running at 0.64 FPS.
[Genesis] [21:08:52] [INFO] Running at 0.67 FPS.
[Genesis] [21:08:53] [INFO] Running at 0.69 FPS.
[Genesis] [21:08:54] [INFO] Running at 0.71 FPS.
[Genesis] [21:08:55] [INFO] Running at 0.73 FPS.
[Genesis] [21:08:56] [INFO] Running at 0.75 FPS.
[Genesis] [21:08:56] [INFO] Running at 0.77 FPS.
[Genesis] [21:08:57] [INFO] Running at 0.79 FPS.
[Genesis] [21:08:58]